# Working with DLIS Files

DLIS (Digital Log Interchange Standard) is a binary well log format commonly used in the oil & gas industry. Unlike LAS files, DLIS files can contain:

- Multiple logical files per physical file
- Multiple frames (data tables) per logical file
- Multi-dimensional curve data
- Rich metadata about the well and logging operation

Welly supports reading DLIS files through the `Well.from_dlis()` method.

## Installation

DLIS support requires the `dlisio` library. Install it with:

```bash
pip install welly[dlis]
```

Or install dlisio separately:

```bash
pip install dlisio
```

### Sample DLIS Files

If you need DLIS files for testing, the Utah FORGE project provides a collection of well log data including DLIS files with FMI images, CBL data, and standard logs:

https://gdr.openei.org/submissions/1330

In [ ]:
import welly
from welly import Well

## Inspecting a DLIS File

Before loading data, you can inspect what's in a DLIS file using `welly.describe_dlis()`. This is useful because DLIS files can contain multiple frames with different data:

In [ ]:
# Inspect a DLIS file (replace with your file path)
# info = welly.describe_dlis('path/to/your/file.dlis')
# 
# print(f"Well: {info['logical_files'][0]['well_name']}")
# print(f"Company: {info['logical_files'][0]['company']}")
# print(f"Tools: {[t['name'] for t in info['logical_files'][0]['tools']]}")
# 
# print("\nFrames:")
# for frame in info['logical_files'][0]['frames']:
#     print(f"  {frame['name']}: {frame['n_curves']} curves")
#     print(f"    Curves: {frame['curves'][:5]}...")

## Loading a DLIS File

The simplest way to load a DLIS file is to use `Well.from_dlis()` with just the filename. This loads the first frame from the first logical file:

In [ ]:
# Load a DLIS file (replace with your file path)
# well = Well.from_dlis('path/to/your/file.dlis')
# well

## Loading Specific Frames

DLIS files can contain multiple frames with different sampling rates or curve sets. Use `describe_dlis()` to find the frame you need, then load it by name:

In [ ]:
# Load a specific frame by name
# well = Well.from_dlis('file.dlis', frame='60B')

## Loading from Different Logical Files

DLIS files can contain multiple logical files (e.g., different logging runs). Use the `logical_file` parameter to select which one:

In [ ]:
# Load from the second logical file (index 1)
# well = Well.from_dlis('file.dlis', logical_file=1)

## Loading All Wells from a File

To load all wells (all frames from all logical files), use `return_all=True`:

In [ ]:
# Load all wells from the file
# wells = Well.from_dlis('file.dlis', return_all=True)
# print(f"Loaded {len(wells)} wells")
# for w in wells:
#     print(f"  {w.name}: {len(w.data)} curves, frame={w._dlis_frame}")

## Error Handling

DLIS files can sometimes be malformed. The `error_handling` parameter controls how errors are handled:

- `'warn'` (default): Log warnings but continue parsing
- `'strict'`: Raise exceptions on errors
- `'ignore'`: Silently ignore errors

In [ ]:
# Lenient error handling for problematic files
# well = Well.from_dlis('file.dlis', error_handling='ignore')

## DLIS Metadata

DLIS wells store additional metadata about the source file and logging tools:

- `well._dlis_frame`: Name of the frame the data came from
- `well._dlis_frame_description`: Description of the frame
- `well._dlis_index_type`: Type of index (e.g., 'BOREHOLE-DEPTH')
- `well._dlis_logical_file`: Index of the logical file
- `well._dlis_tools`: List of logging tools used

In [ ]:
# Access DLIS metadata
# print(f"Frame: {well._dlis_frame}")
# print(f"Description: {well._dlis_frame_description}")
# print(f"Index type: {well._dlis_index_type}")
# print(f"Tools:")
# for tool in well._dlis_tools:
#     print(f"  - {tool['name']}: {tool['description']}")

## Working with DLIS Wells

Once loaded, DLIS wells work exactly like LAS wells. You can:

- Access curves via `well.data`
- Get DataFrames with `well.df()`
- Plot curves
- Export to LAS format

In [ ]:
# Example workflow (uncomment with a real file)
# well = Well.from_dlis('file.dlis')
# 
# # View available curves
# print("Curves:", list(well.data.keys()))
# 
# # Get a DataFrame
# df = well.df(keys=['GR', 'RHOB', 'NPHI'])
# print(df.head())
# 
# # Plot a curve
# well.data['GR'].plot()
# 
# # Export to LAS
# well.to_las('output.las')

## Example: Finding CBL Data

Here's a practical example of finding and loading Cement Bond Log (CBL) data from a DLIS file:

In [ ]:
# # 1. Inspect the file
# info = welly.describe_dlis('cbl_file.dlis')
# 
# # 2. Find frames with CBL curves
# for frame in info['logical_files'][0]['frames']:
#     cbl_curves = [c for c in frame['curves'] if 'CBL' in c]
#     if cbl_curves:
#         print(f"Frame '{frame['name']}' has CBL curves: {cbl_curves}")
# 
# # 3. Load the frame with CBL data
# well = Well.from_dlis('cbl_file.dlis', frame='60B')
# 
# # 4. Access the CBL curve
# cbl = well.data['CBL']
# cbl.plot()

## Borehole Image Logs (FMI, UBI, etc.)

DLIS files can contain 2D borehole image data from tools like FMI (Formation MicroImager), UBI (Ultrasonic Borehole Imager), and similar. These images have:

- Rows corresponding to depth samples
- Columns corresponding to azimuthal samples (typically 0-360 degrees)

Welly provides the `ImageCurve` class and `welly.load_images()` function for working with this data.

### Loading Image Data

Use `welly.load_images()` to load 2D image data from a DLIS file:

In [ ]:
# Load image data from a DLIS file
# images = welly.load_images('fmi_data.dlis')
# 
# # See what images are available
# print("Available images:", list(images.keys()))
# 
# # Get an image
# fmi = images['FMI_DYN']
# print(fmi)

### ImageCurve Properties

The `ImageCurve` class provides useful properties for understanding your image data:

In [ ]:
# # Image properties
# print(f"Mnemonic: {fmi.mnemonic}")
# print(f"Shape: {fmi.shape}")
# print(f"Depth range: {fmi.start:.1f} - {fmi.stop:.1f} {fmi.index_units}")
# print(f"Number of azimuths: {fmi.n_azimuths}")
# print(f"Number of depth samples: {fmi.n_samples}")
# print(f"Units: {fmi.units}")

### Quick Visualization

Use `plot()` for a quick visualization of the image. The x-axis shows azimuth with cardinal direction labels (N/E/S/W):

In [ ]:
# # Quick plot of a depth section
# section = fmi.get_section(5000, 5050)  # 50 ft section
# section.plot(cmap='YlOrBr')

### Customizing Plots

The `plot()` method accepts several parameters for customization:

In [ ]:
# # Customized plot
# import matplotlib.pyplot as plt
# 
# fig, ax = plt.subplots(figsize=(10, 12))
# section.plot(
#     ax=ax,
#     cmap='viridis',           # Different colormap
#     percentile_clip=(2, 98),  # Adjust color scaling
#     show_colorbar=True,
#     title='FMI Image - 5000-5050 ft'
# )
# plt.tight_layout()
# plt.show()

### Multi-Page PDF Export

For full-well images, use `to_pdf()` to create a multi-page PDF similar to traditional well log prints. This is the recommended approach for viewing complete image logs:

In [ ]:
# # Export to multi-page PDF
# n_pages = fmi.to_pdf(
#     'fmi_output.pdf',
#     feet_per_page=100,    # 100 ft per page
#     cmap='YlOrBr',
#     dpi=100,
#     show_progress=True
# )
# print(f"Created {n_pages} page PDF")

### PNG Series Export

Alternatively, export as a series of PNG files:

In [ ]:
# # Export as PNG series
# filenames = fmi.to_png_series(
#     'output_images/',
#     prefix='fmi',
#     feet_per_image=100,
#     dpi=150
# )
# print(f"Created {len(filenames)} PNG files")

### Extracting Depth Sections

Use `get_section()` to extract a specific depth interval:

In [ ]:
# # Extract a section
# zone_of_interest = fmi.get_section(5200, 5250)
# print(f"Section: {zone_of_interest.start:.1f} - {zone_of_interest.stop:.1f} ft")
# print(f"Samples: {zone_of_interest.n_samples}")
# 
# # Plot the section
# zone_of_interest.plot(title='Zone of Interest')

### Working with Image Data Directly

The raw image data is available as a NumPy array for custom analysis:

In [ ]:
# import numpy as np
# 
# # Access raw data
# data = fmi.data  # 2D numpy array
# depths = fmi.index  # 1D depth array
# 
# # Example: compute statistics
# print(f"Data range: {np.nanmin(data):.2f} - {np.nanmax(data):.2f}")
# print(f"Mean: {np.nanmean(data):.2f}")
# print(f"Std: {np.nanstd(data):.2f}")

### De-rotating Images (Aligning Pads Vertically)

Borehole imaging tools like FMI have multiple pads that don't cover the full 360° of the borehole. As the tool moves down the hole, it rotates, causing the gaps between pads to appear as diagonal white lines in the image.

If an orientation curve (like P1NO - Pad 1 North Offset) is available in the DLIS file, welly can de-rotate the image to align the pads vertically. This makes it easier to see continuous features within each pad.

In [ ]:
# # Check if de-rotation is available
# print(f"Can de-rotate: {fmi.can_derotate}")
# 
# # De-rotate the image
# if fmi.can_derotate:
#     fmi_derotated = fmi.derotate()
#     print(f"De-rotated: {fmi_derotated}")

### Side-by-Side Comparison

Use `plot_with_derotated()` to see both views together:

In [ ]:
# # Plot original and de-rotated side by side
# section = fmi.get_section(5000, 5050)
# fig, axes = section.plot_with_derotated()
# plt.show()

### Dual-View PDF Export

Export both views to a multi-page PDF:

In [ ]:
# # Export both views to PDF
# n_pages = fmi.to_pdf_with_derotated(
#     'fmi_comparison.pdf',
#     feet_per_page=100,
#     show_progress=True
# )
# print(f"Created {n_pages} page comparison PDF")

**Note:** The de-rotated view shows the image with Pad 1 always at 0°. The x-axis labels change from cardinal directions (N/E/S/W) to pad positions (P1, P3, P5, P7). This view is useful for:

- Seeing continuous features within each pad
- Quality control of individual pad data
- Identifying pad-specific issues

The original geographic north view is still important for:

- Interpreting dip direction of features
- Correlating with other directional data
- Standard geological interpretation

## Summary

Key points for working with DLIS files:

1. Install with `pip install welly[dlis]`
2. Use `welly.describe_dlis()` to inspect file contents first
3. Use `Well.from_dlis()` to load 1D curve data
4. Specify `frame='name'` to load a specific frame
5. Use `return_all=True` to get all wells from a file
6. Access tool metadata via `well._dlis_tools`
7. Once loaded, DLIS wells work like any other welly Well

For borehole images:

8. Use `welly.load_images()` to load 2D image data (FMI, UBI, etc.)
9. Use `ImageCurve.plot()` for quick visualization
10. Use `ImageCurve.to_pdf()` for multi-page PDF export (recommended for full wells)
11. Use `ImageCurve.get_section()` to extract depth intervals
12. Use `ImageCurve.derotate()` to align pads vertically (if orientation data available)
13. Use `ImageCurve.plot_with_derotated()` for side-by-side comparison